<center>

# **Delete and Purge workspaces**

</center>  

### Purpose
This notebook permanently deletes (purges) Fabric workspaces by name using a non-documented internal Microsoft endpoint.

**_Disclaimer:_** This solution uses a non-documented and internal Microsoft endpoint for purging (permanently deleting) workspaces. Since this is not an officially supported API, it may change without notice, which could impact the functionality of this approach. Use it with that in mind, and feel free to experiment!
Also these APIs should be run in the context of a User Principal as support for Service Principal on internal endpoints are not guaranteed.

<mark>**USE WITH CAUTION - PERMANENTLY DELETED WORKSPACES CANNOT BE RECOVERED!**</mark>


### Configuration
Specify the workspaces to permanently delete and whether to run in dry-run mode.
- **workspace_names**  
  Array of workspace names to permanently delete. Names are matched case-insensitively against the `name` property returned from the admin workspaces endpoint. If a name matches multiple workspaces, all matches are included.
- **is_dryrun**  
  When `True`, the notebook only previews which workspaces would be deleted. Set to `False` to actually perform the permanent deletion.

In [ ]:
workspace_names = [
    "<workspace_name_1>",
    "<workspace_name_2>",
]

is_dryrun = True

# Optional: paste a full JWT here to override the token from notebookutils.
# Must be the complete token. If you copied it from a UI that abbreviates with "..." (U+2026)
# the notebook will refuse it - fetch the full token from a source that doesn't truncate.
manual_token = None

### Notebook functionality
The cells below define helper functions, fetch the list of workspaces from the admin metadata endpoint, resolve the configured names to IDs, and (unless in dry-run mode) call the `deleteWorkspaces` endpoint to permanently delete them.

In [ ]:
import requests, re, time, uuid
from html import escape as _h


def _resolve_token():
    raw = manual_token if manual_token else notebookutils.credentials.getToken("pbi")
    raw = (raw or "").strip()
    if not raw:
        raise RuntimeError("No access token available - set manual_token or check notebookutils.credentials.getToken.")
    try:
        raw.encode("ascii")
    except UnicodeEncodeError as ex:
        bad = raw[ex.start:ex.end]
        raise RuntimeError(
            f"Token contains non-ASCII char {bad!r} (U+{ord(bad):04X}) at position {ex.start}. "
            "This usually means the token was copied from a UI that truncated it with an ellipsis. "
            "Re-copy the full JWT (no '...') and try again."
        ) from None
    return raw


def _headers(admin: bool = False):
    h = {
        "Authorization": f"Bearer {_resolve_token()}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/plain, */*",
        "ActivityId": str(uuid.uuid4()),
        "RequestId": str(uuid.uuid4()),
    }
    if admin:
        # Admin metadata endpoints route on this header; without it they 404 instead of 403.
        h["X-PowerBI-User-Admin"] = "true"
    return h


### Get cluster URL for use in metadata endpoints (unsupported endpoints)
def get_cluster_url():
    for attempt in range(2):  # initial attempt + 1 retry
        response = requests.get("https://api.powerbi.com/v1.0/myorg/capacities", headers=_headers())
        response.raise_for_status()
        match = re.match(r"(https://[^/]+/)", response.json().get("@odata.context", ""))
        if match and "redirect.analysis.windows.net" in match.group(1):
            return match.group(1)

        response = requests.get("https://api.powerbi.com/v1.0/myorg/datasets", headers=_headers())
        response.raise_for_status()
        match = re.match(r"(https://[^/]+/)", response.json().get("@odata.context", ""))
        if match and "redirect.analysis.windows.net" in match.group(1):
            return match.group(1)

        if attempt == 0:
            time.sleep(1)
    return None


def get_admin_workspaces(cluster_base_url, top: int = 100000, skip: int = 0, include_users: bool = False):
    # The endpoint requires top/skip/includeUsers query params - it 404s without them.
    params = {"top": top, "skip": skip, "includeUsers": str(include_users).lower()}
    response = requests.get(
        f"{cluster_base_url}metadata/admin/workspaces",
        headers=_headers(admin=True),
        params=params,
    )
    if not response.ok:
        print(f"GET {response.url}\n  status: {response.status_code}\n  body: {response.text[:500]}")
        response.raise_for_status()
    return response.json().get("workspaces", [])


def delete_workspace(workspace_id):
    # Standard Fabric workspace delete - soft-deletes the workspace (Active -> Deleted).
    response = requests.delete(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}",
        headers=_headers(),
    )
    if not response.ok:
        print(f"DELETE {response.url}\n  status: {response.status_code}\n  body: {response.text[:500]}")
        response.raise_for_status()
    return response


def delete_workspaces_permanently(cluster_base_url, workspace_ids):
    response = requests.post(
        f"{cluster_base_url}metadata/admin/workspaces/deleteWorkspaces",
        headers=_headers(admin=True),
        json=workspace_ids,
    )
    if not response.ok:
        print(f"POST {response.url}\n  status: {response.status_code}\n  body: {response.text[:500]}")
        response.raise_for_status()
    return response


def _workspaces_table_html(workspaces, header, header_color=None):
    color_attr = f" style='color:{header_color}'" if header_color else ""
    if not workspaces:
        return f"<h4{color_attr}>{_h(header)} (0)</h4><i>None.</i>"
    rows = "".join(
        f"<tr>"
        f"<td style='padding:4px 12px'>{_h(str(w.get('name','')))}</td>"
        f"<td style='padding:4px 12px'><code>{_h(str(w.get('id','')))}</code></td>"
        f"<td style='padding:4px 12px'>{_h(str(w.get('type','')))}</td>"
        f"<td style='padding:4px 12px'>{_h(str(w.get('state','')))}</td>"
        f"<td style='padding:4px 12px'>{_h(str((w.get('capacity') or {}).get('displayName','')))}</td>"
        f"</tr>"
        for w in workspaces
    )
    return (
        f"<h4{color_attr}>{_h(header)} ({len(workspaces)})</h4>"
        "<table style='border-collapse:collapse'>"
        "<tr style='text-align:left;border-bottom:1px solid #888'>"
        "<th style='padding:4px 12px'>Workspace name</th>"
        "<th style='padding:4px 12px'>Workspace ID</th>"
        "<th style='padding:4px 12px'>Type</th>"
        "<th style='padding:4px 12px'>State</th>"
        "<th style='padding:4px 12px'>Capacity</th>"
        f"</tr>{rows}</table>"
    )


def render_workspaces_table(workspaces, header):
    displayHTML(_workspaces_table_html(workspaces, header))


def render_not_found(names):
    if not names:
        return
    rows = [{"name": n} for n in names]
    displayHTML(_workspaces_table_html(rows, "Workspace names not found", header_color="#b00"))

In [ ]:
### Set cluster url based on response from Power BI API call. Overwrite with manual value if required.
CLUSTER_BASE_URL = get_cluster_url()
print(f"Using base URL: {CLUSTER_BASE_URL}")

### Fetch all workspaces and resolve names to IDs
Calls `/metadata/admin/workspaces` to get every workspace (id, name, type, state, capacity) and builds a case-insensitive lookup. Each configured name is then resolved to one or more workspaces; names with no match are collected separately.

In [ ]:
all_workspaces = get_admin_workspaces(CLUSTER_BASE_URL)
print(f"Fetched {len(all_workspaces)} workspaces from /metadata/admin/workspaces")

# Build a case-insensitive name -> workspaces lookup. A name may match multiple workspaces (e.g. personal workspaces); keep all matches.
by_name = {}
for ws in all_workspaces:
    by_name.setdefault((ws.get("name") or "").lower(), []).append(ws)

matched = []
not_found = []
for name in workspace_names:
    hits = by_name.get(name.lower(), [])
    if hits:
        matched.extend(hits)
    else:
        not_found.append(name)

print(f"Matched: {len(matched)} | Not found: {len(not_found)}")

### Permanently delete the matched workspaces
In dry-run mode the table below shows the workspaces that *would* be permanently deleted. With `is_dryrun = False` the notebook calls `/metadata/admin/workspaces/deleteWorkspaces` with the resolved IDs and then displays the workspaces that were deleted. Names that could not be resolved are listed separately.

In [ ]:
if is_dryrun:
    print("\033[1mDry run mode\033[0m - no workspaces will be deleted.")
    render_workspaces_table(matched, "Workspaces to be permanently deleted (dry run)")
elif not matched:
    print("No matched workspaces - nothing to delete.")
else:
    # Step 1: soft-delete any workspaces still in Active state via the standard Fabric REST API.
    #         The admin purge endpoint only operates on already-deleted workspaces.
    active = [w for w in matched if (w.get("state") or "").lower() == "active"]
    if active:
        print(f"Soft-deleting {len(active)} active workspace(s) via DELETE /v1/workspaces/{{id}}...")
        for w in active:
            delete_workspace(w["id"])
            print(f"  - {w.get('name')} ({w['id']}) -> Deleted")
    else:
        print("No active workspaces among the matches - skipping soft-delete step.")

    # Step 2: permanently delete (purge) all matched workspaces.
    ids = [w["id"] for w in matched]
    response = delete_workspaces_permanently(CLUSTER_BASE_URL, ids)
    render_workspaces_table(matched, "Workspaces permanently deleted")

render_not_found(not_found)